# 01 - Cross-validation and comparison of nine models on MSLD v2.0

Code for the paper *A comparative analysis of convolutional, transformer and hybrid deep learning models for Mpox
skin lesion diagnosis under leakage-free patient-level evaluation*.

Nine pretrained models are trained with one recipe and evaluated on the same patient-level
five-fold cross-validation folds of the Mpox Skin Lesion Dataset v2.0 (755 original images,
524 patients, six classes). All backbones are frozen; only the classification head is trained.

| Name in the code | Name in the paper | Family |
|---|---|---|
| `MobileNetV2` | MobileNetV2 | CNN |
| `EfficientNetV2` | EfficientNetV2-B0 | CNN |
| `ResNet50` | ResNet50 | CNN |
| `DenseNet121` | DenseNet121 | CNN |
| `ViT-B/16` | ViT-B/16 (ImageNet-21k, supervised) | Transformer |
| `DINO ViT-B/16` | DINO ViT-B/16 (ImageNet-1k, self-supervised) | Transformer |
| `Hybrid+Adam` | Hybrid + Adam | Hybrid (MobileNetV2 + ViT-B/16) |
| `MoViT-SAM` | Hybrid + SAM, the reference model | Hybrid |
| `Hybrid+H-SAM` | Hybrid + H-SAM | Hybrid |

| Section | What it does |
|---|---|
| 0-1 | Dependencies, configuration, random seeds |
| 2-4 | Dataset, patient identifiers from filenames, near-duplicate and skin-tone audit |
| 5 | Patient-level 70/15/15 split (used by notebook 02) |
| 6-8 | Input pipeline, focal loss with label smoothing, Mixup, SAM and H-SAM |
| 9-10b | The nine models |
| 11 | Five-fold stratified-group cross-validation, out-of-fold predictions for every model |
| 12-15 | Reference model: fold summary, per-class results, calibration, screening, decision curve |
| 14b | All models: confidence intervals, patient-level accuracy, calibration, Friedman test, pairwise McNemar tests with Holm correction |
| 16-16b | Leakage protocols A and B for a hybrid, a CNN and a transformer |
| 17 | Zip of all result files |

**Safeguards enforced in the code.** Splits are disjoint by patient. Augmentation is applied only to
training batches. Early stopping and learning-rate scheduling use an inner validation split taken
from the training patients, never the evaluation fold. Temperature scaling is cross-fitted, so no
prediction is calibrated with a temperature fitted on itself.

**Running on Kaggle.** Attach MSLD v2.0 as an input, or turn Internet on so that Section 2 can
download it, and select a GPU. For a quick end-to-end check set `SMOKE_TEST = True` in Section 1:
every model is then trained for one epoch on two folds. For the full experiment leave
`SMOKE_TEST = False` and use *Save Version > Save & Run All*; it takes roughly nine hours on two
T4 GPUs. Each fold is checkpointed, so an interrupted session resumes where it stopped.

Results are written to `./results/` and `./results/comparative/`, and packed into
`comparative_results.zip`.

## Setup

In [ ]:
# ============================================================
# 0 - Install dependencies (Kaggle: Internet = ON)
# ============================================================
import os, subprocess, sys
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # Keras-2 API required by TFViTModel

def pip(args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args,
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("pip FAILED for", args, "\n", (r.stderr or r.stdout)[-1500:])

if not os.environ.get("MOVIT_SKIP_INSTALL"):
    # tf-keras must match the installed TensorFlow (an unpinned install pulls a newer TensorFlow).
    from importlib.metadata import version as _v
    _tf = None
    for _name in ("tensorflow", "tensorflow-cpu"):
        try: _tf = _v(_name); break
        except Exception: pass
    if _tf:
        _mm = ".".join(_tf.split(".")[:2])
        pip([f"tf-keras=={_mm}.*"])
    else:
        pip(["tf-keras"])
    # transformers 4.46.3 still provides TFViTModel and has pre-built wheels (tokenizers 0.20.3) for
    # current Python versions, including 3.13, which Kaggle now uses.
    pip(["transformers==4.46.3", "tokenizers==0.20.3", "huggingface_hub==0.24.6"])
    for p in ["scikit-learn>=1.3.0", "matplotlib>=3.7.0", "seaborn>=0.12.0",
              "opencv-python>=4.8.0", "pandas>=2.0.0", "scipy>=1.11.0",
              "ImageHash>=4.3.1", "kaggle", "Pillow"]:
        pip([p])
    r = subprocess.run([sys.executable, "-c", "from transformers import TFViTModel; import transformers; print(transformers.__version__)"],
                       capture_output=True, text=True, env={**os.environ, "TF_USE_LEGACY_KERAS": "1", "TF_CPP_MIN_LOG_LEVEL": "3"})
    if r.returncode != 0:
        raise RuntimeError("TFViTModel import check failed:\n" + r.stderr[-2000:])
    print("Dependencies installed. transformers", r.stdout.strip().splitlines()[-1], "| TFViTModel import OK")

In [ ]:
# ============================================================
# 1 - Imports, reproducibility, configuration
# ============================================================
import os
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")
import re, random, json, time, platform, itertools
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
import imagehash
from scipy import stats as sp_stats
from scipy.optimize import minimize_scalar

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.model_selection import (StratifiedGroupKFold, GroupShuffleSplit,
                                     StratifiedShuffleSplit)
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve,
                             roc_auc_score, precision_recall_curve,
                             average_precision_score, f1_score)
from sklearn.utils.class_weight import compute_class_weight

sns.set_style("whitegrid")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# TINY_TEST: pipeline check on synthetic images with tiny random backbones (no downloads).
# SMOKE_TEST: real data and backbones, very short training.  Both False = full experiment.
TINY_TEST  = bool(int(os.environ.get("MOVIT_TINY_TEST", "0")))
SMOKE_TEST = False

# Label order. Monkeypox MUST stay at index 0: the exported TFLite model and the
# cross-dataset check read P(Monkeypox) from output index 0.
CLASS_NAMES = ["Monkeypox", "Chickenpox", "Measles", "Cowpox", "HFMD", "Healthy"]
NUM_CLASSES = len(CLASS_NAMES)
MPOX_IDX    = CLASS_NAMES.index("Monkeypox")
IMG_SIZE, CHANNELS = 224, 3

TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
INNER_VAL_FRAC = 0.15            # inner early-stopping split inside each training fold

quick = TINY_TEST or SMOKE_TEST
BATCH_SIZE = 16 if quick else 32
EPOCHS     = 1  if quick else 40
PATIENCE   = 1  if quick else 8
K_FOLDS    = 2  if quick else 5
LR_HEAD, LR_MOBILENET, LR_VIT, LR_EFFNET = 5e-4, 1e-3, 1e-4, 1e-3

FOCAL_GAMMA     = 2.0
LABEL_SMOOTHING = 0.05
MIXUP_ALPHA     = 0.2
RHO, ALPHA      = 0.05, 0.5      # SAM radius; H-SAM adaptive coefficient

N_BOOT          = 200 if quick else 2000
PREVALENCES     = [0.01, 0.05, 0.15]          # + dataset prevalence (added below)
SENS_TARGETS    = [0.90, 0.95]
DCA_THRESHOLDS  = np.arange(0.01, 0.601, 0.01)
PHASH_THRESHOLD = 8
AUG_COPIES_B    = 6                           # Protocol B: copies per original

DATA_ROOT   = Path(os.environ.get("MOVIT_DATA_ROOT", "./data/msld_v2"))
RESULTS_DIR = Path(os.environ.get("MOVIT_RESULTS", "./results")); RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR  = Path(os.environ.get("MOVIT_MODELS", "./saved_models")); MODELS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR    = RESULTS_DIR / "cv_checkpoints"; CKPT_DIR.mkdir(exist_ok=True)

gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass

HARDWARE = {
    "python": platform.python_version(), "tensorflow": tf.__version__,
    "platform": platform.platform(), "processor": platform.processor() or platform.machine(),
    "cpu_count": os.cpu_count(), "gpu": [g.name for g in gpus] or "none",
}
from importlib.metadata import version as _ver
for _pkg in ("transformers", "tf-keras", "scikit-learn", "numpy"):
    try: HARDWARE[_pkg] = _ver(_pkg)
    except Exception: pass
try:
    HARDWARE["cpu_model"] = next(l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo")
                                 if l.startswith("model name"))
except Exception:
    pass
(RESULTS_DIR / "hardware.json").write_text(json.dumps(HARDWARE, indent=2))
if not gpus and not (TINY_TEST or SMOKE_TEST):
    raise RuntimeError("No GPU detected. On Kaggle: Settings > Accelerator > GPU T4 x2, then Save & Run All again. "
                       "Without a GPU the full run would take days.")
print(json.dumps(HARDWARE, indent=2))
print("TINY_TEST:", TINY_TEST, "| SMOKE_TEST:", SMOKE_TEST, "| classes:", CLASS_NAMES)

## Data: MSLD v2.0 originals, patient identifiers and integrity audit

In [ ]:
# ============================================================
# 2 - Obtain MSLD v2.0
# ============================================================
# Dataset: https://www.kaggle.com/datasets/joydippaul/mpox-skin-lesion-dataset-version-20-msld-v20
# On Kaggle, add it via "Add Input" and point MOVIT_DATA_ROOT (or DATA_ROOT) at it.
KAGGLE_SLUG = "joydippaul/mpox-skin-lesion-dataset-version-20-msld-v20"

def _has_images(root):
    return root.exists() and any(p.suffix.lower() in {".jpg", ".jpeg", ".png"} for p in root.rglob("*"))

if TINY_TEST and not _has_images(DATA_ROOT):
    # Synthetic stand-in with MSLD-style filenames (class prefix + patient + image no.).
    rng = np.random.default_rng(SEED)
    prefix = {"Monkeypox": "MKP", "Chickenpox": "CHP", "Measles": "MSL",
              "Cowpox": "CWP", "HFMD": "HFMD", "Healthy": "HEALTHY"}
    for ci, cls in enumerate(CLASS_NAMES):
        d = DATA_ROOT / "Original Images" / cls; d.mkdir(parents=True, exist_ok=True)
        for pt in range(1, 9):
            base = rng.integers(40, 200, 3)
            for k in range(1, 3):
                img = np.clip(base + rng.normal(0, 25, (96, 96, 3)) + ci * 8, 0, 255).astype("uint8")
                Image.fromarray(img).save(d / f"{prefix[cls]}_{pt:02d}_{k:02d}.jpg")
elif not _has_images(DATA_ROOT):
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    rc = os.system(f'kaggle datasets download -d {KAGGLE_SLUG} --unzip -p "{DATA_ROOT}"')
    if rc != 0:
        raise RuntimeError("Download failed: attach the dataset as a Kaggle input and set DATA_ROOT.")

In [ ]:
# ============================================================
# 3 - Locate ORIGINAL images, build catalogue, extract patient IDs
# ============================================================
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
CLASS_ALIASES = {
    "monkeypox": "Monkeypox", "mpox": "Monkeypox", "mkp": "Monkeypox", "monkey pox": "Monkeypox",
    "chickenpox": "Chickenpox", "chicken pox": "Chickenpox", "chp": "Chickenpox",
    "measles": "Measles", "msl": "Measles",
    "cowpox": "Cowpox", "cow pox": "Cowpox", "cwp": "Cowpox",
    "hfmd": "HFMD", "hand foot mouth disease": "HFMD", "handfootmouth": "HFMD",
    "healthy": "Healthy", "normal": "Healthy", "hth": "Healthy",
}

def canonical(name):
    key = re.sub(r"\s+", " ", name.strip().lower().replace("-", " ").replace("_", " "))
    return CLASS_ALIASES.get(key) or CLASS_ALIASES.get(key.replace(" ", ""))

def extract_patient_id(cls, fname):
    """MSLD filenames encode the patient: e.g. MKP_01_01.jpg -> Monkeypox_0001.
    Images without a parsable patient number are treated as their own patient."""
    m = re.match(r"^([A-Za-z]+)[ _-]?(\d+)", Path(fname).stem)
    return f"{cls}_{int(m.group(2)):04d}" if m else f"{cls}_{Path(fname).stem}"

# MSLD v2.0 distributes its ORIGINAL images across several fold sub-folders (e.g. Train /
# Valid / Test inside each fold), so no single folder holds all of them. We therefore walk the
# whole dataset, skip anything under an "augmented" path, take the class from the nearest
# class-named ancestor folder, and de-duplicate by (class, filename) to recover the union of
# unique originals.
EXPECTED_ORIGINALS = 755   # MSLD v2.0 original images

def class_of(path):
    for parent in path.parents:
        c = canonical(parent.name)
        if c: return c
        if parent == DATA_ROOT: break
    return None

records = []
for f in sorted(DATA_ROOT.rglob("*")):
    if f.suffix.lower() not in IMG_EXT or "augment" in str(f).lower():
        continue
    cls = class_of(f)
    if cls is None: continue
    records.append({"path": str(f), "fname": f.name.lower(), "cls": cls,
                    "label": CLASS_NAMES.index(cls), "patient": extract_patient_id(cls, f.name)})
df = (pd.DataFrame(records).drop_duplicates(subset=["cls", "fname"])
        .sort_values(["label", "fname"]).reset_index(drop=True))
paths, labels, groups = df["path"].values, df["label"].values, df["patient"].values
print(df.groupby("cls").agg(images=("path", "size"), patients=("patient", "nunique"))
        .reindex(CLASS_NAMES).to_string())
print(f"TOTAL: {len(df)} images, {df['patient'].nunique()} patients")
if not TINY_TEST and len(df) != EXPECTED_ORIGINALS:
    raise RuntimeError(f"Found {len(df)} unique original images, expected {EXPECTED_ORIGINALS}. "
                       "Check DATA_ROOT points at the full MSLD v2.0 dataset before running.")

# Invalidate saved CV folds if the image set has changed since they were written.
import hashlib, shutil
FINGERPRINT = hashlib.sha256("\n".join(sorted(df["cls"] + "/" + df["fname"])).encode()).hexdigest()
_fp = CKPT_DIR / "dataset_fingerprint.txt"
if _fp.exists() and _fp.read_text().strip() != FINGERPRINT:
    print("Dataset changed since the saved CV folds were written: deleting old checkpoints.")
    shutil.rmtree(CKPT_DIR); CKPT_DIR.mkdir()
_fp.write_text(FINGERPRINT)

In [ ]:
# ============================================================
# 4 - Data integrity audit: perceptual hashing and skin tone (ITA)
# ============================================================
df["phash"] = [imagehash.phash(Image.open(p).convert("RGB")) for p in df["path"]]

def near_duplicate_pairs(hashes, thr=PHASH_THRESHOLD):
    h = list(hashes)
    return [(i, j, h[i] - h[j]) for i in range(len(h)) for j in range(i + 1, len(h))
            if h[i] - h[j] <= thr]

dup_pairs = near_duplicate_pairs(df["phash"])
cross_patient = [(i, j, d) for i, j, d in dup_pairs if groups[i] != groups[j]]
print(f"Near-duplicate pairs (Hamming <= {PHASH_THRESHOLD}): {len(dup_pairs)} "
      f"({len(cross_patient)} across different inferred patients)")
pd.DataFrame([{"a": paths[i], "b": paths[j], "hamming": d, "same_patient": groups[i] == groups[j]}
              for i, j, d in dup_pairs]).to_csv(RESULTS_DIR / "audit_near_duplicates.csv", index=False)

def ita_degrees(path):
    """Individual Typology Angle, ITA = arctan((L* - 50) / b*) in degrees, taken as the median
    over skin-coloured pixels (YCrCb skin mask, Cr 133-173, Cb 77-127). A proxy for skin tone,
    not a validated Fitzpatrick assessment."""
    rgb = np.array(Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
    ycc = cv2.cvtColor(rgb, cv2.COLOR_RGB2YCrCb)
    mask = (ycc[..., 1] >= 133) & (ycc[..., 1] <= 173) & (ycc[..., 2] >= 77) & (ycc[..., 2] <= 127)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L = lab[..., 0] * 100.0 / 255.0
    b = lab[..., 2] - 128.0
    sel = mask & (b > 1.0)
    if sel.sum() < 50:
        return np.nan
    return float(np.median(np.degrees(np.arctan((L[sel] - 50.0) / b[sel]))))

df["ita"] = [ita_degrees(p) for p in df["path"]]
ita_tab = df.groupby("cls")["ita"].agg(["median", "count"]).reindex(CLASS_NAMES).round(1)
print("Median ITA by class:\n", ita_tab.to_string())
ita_tab.to_csv(RESULTS_DIR / "audit_ita_by_class.csv")

In [ ]:
# ============================================================
# 5 - Patient-level 70/15/15 split BEFORE augmentation
# ============================================================
gss1 = GroupShuffleSplit(n_splits=1, train_size=TRAIN_RATIO, random_state=SEED)
train_idx, temp_idx = next(gss1.split(paths, labels, groups))
gss2 = GroupShuffleSplit(n_splits=1, train_size=VAL_RATIO / (VAL_RATIO + TEST_RATIO), random_state=SEED)
v_rel, t_rel = next(gss2.split(paths[temp_idx], labels[temp_idx], groups[temp_idx]))
val_idx, test_idx = temp_idx[v_rel], temp_idx[t_rel]

G = [set(groups[i]) for i in (train_idx, val_idx, test_idx)]
assert G[0].isdisjoint(G[1]) and G[0].isdisjoint(G[2]) and G[1].isdisjoint(G[2]), "Patient leak!"

def split_near_dups(a_idx, b_idx):
    ha, hb = df["phash"].values[a_idx], df["phash"].values[b_idx]
    return sum(1 for x in hb if any(x - y <= PHASH_THRESHOLD for y in ha))

print("Patient-disjoint split verified.")
print("Test images with a near-duplicate in train:", split_near_dups(train_idx, test_idx))

def counts(idx):
    c = Counter(labels[idx]); return [c.get(k, 0) for k in range(NUM_CLASSES)]
def pcounts(idx):
    s = defaultdict(set)
    for i in idx: s[labels[i]].add(groups[i])
    return [len(s[k]) for k in range(NUM_CLASSES)]

all_idx = np.arange(len(df))
table1 = pd.DataFrame({"Class": CLASS_NAMES, "Total images": counts(all_idx),
                       "Patients": pcounts(all_idx), "Train images": counts(train_idx),
                       "Val images": counts(val_idx), "Test images": counts(test_idx)})
table1.loc[len(table1)] = ["TOTAL", len(df), df["patient"].nunique(), len(train_idx),
                           len(val_idx), len(test_idx)]
print(table1.to_string(index=False))
print(f"Held-out test set: {len(test_idx)} images from {len(G[2])} patients")
table1.to_csv(RESULTS_DIR / "table1_split.csv", index=False)

## Training recipe shared by every model

In [ ]:
# ============================================================
# 6 - Input pipeline: raw [0,255] RGB, on-the-fly augmentation, class weights, Mixup
# ============================================================
# Every model receives raw [0,255] pixels and applies its backbone's own preprocessing
# internally, so the exported TFLite model also expects raw [0,255] RGB input.
AUTOTUNE = tf.data.AUTOTUNE
_AUG = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    layers.RandomRotation(0.15, fill_mode="reflect", seed=SEED),  # +/-0.15 of a full turn (+/-54 deg)
    layers.RandomZoom(0.15, fill_mode="reflect", seed=SEED),      # +/-15 %
    layers.RandomContrast(0.10, seed=SEED),                       # +/-10 %
], name="augmentation")

def class_weights_for(idx):
    present = np.unique(labels[idx])
    w = compute_class_weight("balanced", classes=present, y=labels[idx])
    cw = {int(c): float(x) for c, x in zip(present, w)}
    return {k: cw.get(k, 1.0) for k in range(NUM_CLASSES)}

def _decode(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=CHANNELS, expand_animations=False)
    img = tf.ensure_shape(img, [None, None, CHANNELS])
    return tf.cast(tf.image.resize(img, [IMG_SIZE, IMG_SIZE]), tf.float32)

def _sample_beta(alpha):
    a = tf.random.gamma([], alpha); b = tf.random.gamma([], alpha)
    return a / (a + b)

def _mixup(x, y, w):
    lam = _sample_beta(MIXUP_ALPHA)
    perm = tf.random.shuffle(tf.range(tf.shape(x)[0]))
    return (lam * x + (1 - lam) * tf.gather(x, perm),
            lam * y + (1 - lam) * tf.gather(y, perm),
            lam * w + (1 - lam) * tf.gather(w, perm))

def make_ds(idx, training=False, cw=None, batch=BATCH_SIZE, path_arr=None, label_arr=None):
    """Training: shuffle -> batch -> augment -> Mixup, class weights as sample weights.
    Evaluation: original images only, in the given order."""
    P = (path_arr if path_arr is not None else paths)[idx]
    Y = (label_arr if label_arr is not None else labels)[idx]
    W = np.array([cw[int(l)] for l in Y], np.float32) if cw else np.ones(len(Y), np.float32)
    ds = tf.data.Dataset.from_tensor_slices((P, Y, W))
    if training:
        ds = ds.shuffle(len(P), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, l, w: (_decode(p), tf.one_hot(l, NUM_CLASSES), w),
                num_parallel_calls=AUTOTUNE).batch(batch)
    if training:
        ds = ds.map(lambda x, y, w: _mixup(_AUG(x, training=True), y, w), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

In [ ]:
# ============================================================
# 7 - Loss: focal loss (gamma = 2) with label smoothing (eps = 0.05)
# ============================================================
# The class-balancing term alpha_t of the focal loss is supplied through the per-sample
# class weights built in Section 6.
def focal_loss(gamma=FOCAL_GAMMA, smoothing=LABEL_SMOOTHING):
    def loss(y_true, y_pred):
        y_true = y_true * (1.0 - smoothing) + smoothing / NUM_CLASSES
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        return tf.reduce_sum(tf.pow(1.0 - y_pred, gamma) * (-y_true * tf.math.log(y_pred)), axis=-1)
    loss.__name__ = "focal_loss"
    return loss

In [ ]:
# ============================================================
# 8 - Sharpness-aware training: plain (Adam) / SAM / H-SAM
# ============================================================
#   SAM  : eps = rho * g / ||g||_2                          (Foret et al., 2021)
#   H-SAM: eps = rho * g / ||g||_2 * (1 + alpha * r),  r = ||theta|| / (||theta|| + ||g||)
class SharpnessAwareModel(keras.Model):
    def __init__(self, inner, mode="sam", rho=RHO, alpha=ALPHA, **kw):
        super().__init__(**kw)
        self.inner, self.mode, self.rho, self.alpha = inner, mode, rho, alpha

    def call(self, x, training=False):
        return self.inner(x, training=training)

    def _perturb(self, grads, vars_):
        gnorm = tf.linalg.global_norm([g for g in grads if g is not None]) + 1e-12
        eps = []
        for g, v in zip(grads, vars_):
            if g is None: eps.append(None); continue
            e = self.rho * tf.cast(g, tf.float32) / gnorm
            if self.mode == "hsam":
                pn = tf.norm(tf.cast(v, tf.float32)) + 1e-12
                gl = tf.norm(tf.cast(g, tf.float32)) + 1e-12
                e = e * (1.0 + self.alpha * pn / (pn + gl))
            eps.append(e)
        return eps

    def train_step(self, data):
        x, y, sw = data
        tv = self.trainable_variables
        with tf.GradientTape() as tape:
            yp = self(x, training=True)
            loss = self.compiled_loss(y, yp, sample_weight=sw, regularization_losses=self.losses)
        g = tape.gradient(loss, tv)
        if self.mode != "none":
            eps = self._perturb(g, tv)
            for v, e in zip(tv, eps):
                if e is not None: v.assign_add(e)
            with tf.GradientTape() as tape2:
                yp2 = self(x, training=True)
                loss2 = self.compiled_loss(y, yp2, sample_weight=sw, regularization_losses=self.losses)
            g = tape2.gradient(loss2, tv)
            for v, e in zip(tv, eps):
                if e is not None: v.assign_sub(e)
        self.optimizer.apply_gradients(zip(g, tv))
        self.compiled_metrics.update_state(y, yp, sample_weight=sw)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        x, y = data[0], data[1]
        sw = data[2] if len(data) > 2 else None
        yp = self(x, training=False)
        self.compiled_loss(y, yp, sample_weight=sw)
        self.compiled_metrics.update_state(y, yp, sample_weight=sw)
        return {m.name: m.result() for m in self.metrics}

## Models

In [ ]:
# ============================================================
# 9 - Backbones (each applies its own preprocessing to raw [0,255] input)
# ============================================================
IMAGENET_MEAN = tf.constant([0.485, 0.456, 0.406])
IMAGENET_STD  = tf.constant([0.229, 0.224, 0.225])

def _tiny_features(dim, name):
    """TINY_TEST stand-in: small random CNN with the same output width (no downloads)."""
    inp = keras.Input((IMG_SIZE, IMG_SIZE, CHANNELS))
    x = layers.Rescaling(1 / 255.)(inp)
    x = layers.Conv2D(8, 5, strides=4, activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(dim, activation="relu")(x)
    return keras.Model(inp, x, name=name)

def mobilenet_features(name="mob"):
    if TINY_TEST: return _tiny_features(1280, name)
    inp = keras.Input((IMG_SIZE, IMG_SIZE, CHANNELS))
    x = keras.applications.mobilenet_v2.preprocess_input(inp)
    bb = keras.applications.MobileNetV2(include_top=False, weights="imagenet",
                                        input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS))
    bb.trainable = False
    x = layers.GlobalAveragePooling2D()(bb(x, training=False))
    return keras.Model(inp, x, name=name)

def effnet_features(name="eff"):
    if TINY_TEST: return _tiny_features(1280, name)
    inp = keras.Input((IMG_SIZE, IMG_SIZE, CHANNELS))
    x = keras.applications.efficientnet_v2.preprocess_input(inp)
    bb = keras.applications.EfficientNetV2B0(include_top=False, weights="imagenet",
                                             input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS))
    bb.trainable = False
    x = layers.GlobalAveragePooling2D()(bb(x, training=False))
    return keras.Model(inp, x, name=name)

def vit_features(name="vit"):
    """Frozen ViT-B/16 -> 768-d CLS token. The backbone is not fine-tuned: only the classification
    head on top is trained (hybrid: 1,052,166 trainable of 89,699,398 parameters)."""
    if TINY_TEST: return _tiny_features(768, name)
    from transformers import TFViTModel
    vit = TFViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
    vit.trainable = False
    inp = keras.Input((IMG_SIZE, IMG_SIZE, CHANNELS))
    # google/vit-base-patch16-224-in21k was trained with mean = std = 0.5 (see its
    # preprocessor_config.json), so the input is scaled to [-1, 1] here, not with the
    # ImageNet mean/std used by the CNN streams.
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0)(inp)
    x = layers.Permute((3, 1, 2))(x)
    return keras.Model(inp, vit(x).last_hidden_state[:, 0, :], name=name)

In [ ]:
# ============================================================
# 10 - Model factory and training helpers
# ============================================================
def _compile(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss=focal_loss(), metrics=["accuracy"])
    return model

def build_single(kind):
    feat, lr = {"mobilenet": (mobilenet_features, LR_MOBILENET),
                "effnet": (effnet_features, LR_EFFNET),
                "vit": (vit_features, LR_VIT)}[kind]
    f = feat()
    x = layers.Dropout(0.3)(layers.Dense(256, activation="relu")(f.output))
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    inner = keras.Model(f.input, out, name=f"{kind}_single")
    m = SharpnessAwareModel(inner, mode="none", name=f"{kind}_adam")
    _ = m(tf.zeros((1, IMG_SIZE, IMG_SIZE, CHANNELS)))
    return _compile(m, lr)

def build_hybrid(mode="sam"):
    """Hybrid: frozen MobileNetV2 (1280) || frozen ViT-B/16 (768) -> concat 2048
    -> Dense(512, ReLU) -> Dropout(0.3) -> Dense(6, softmax). mode: none | sam | hsam."""
    inp = keras.Input((IMG_SIZE, IMG_SIZE, CHANNELS))
    fused = layers.Concatenate(name="fusion")([mobilenet_features("mob_stream")(inp),
                                               vit_features("vit_stream")(inp)])
    h = layers.Dropout(0.3)(layers.Dense(512, activation="relu", name="fc1")(fused))
    out = layers.Dense(NUM_CLASSES, activation="softmax", name="out")(h)
    m = SharpnessAwareModel(keras.Model(inp, out, name=f"hybrid_{mode}"), mode=mode,
                            name=f"hybrid_{mode}_wrapper")
    _ = m(tf.zeros((1, IMG_SIZE, IMG_SIZE, CHANNELS)))
    return _compile(m, LR_HEAD)

MODELS = {   # name -> builder.  MoViT-SAM (the hybrid trained with SAM) is the reference model.
    "MoViT-SAM":      lambda: build_hybrid("sam"),
    "Hybrid+Adam":    lambda: build_hybrid("none"),
    "Hybrid+H-SAM":   lambda: build_hybrid("hsam"),
    "EfficientNetV2": lambda: build_single("effnet"),
    "MobileNetV2":    lambda: build_single("mobilenet"),
    "ViT-B/16":       lambda: build_single("vit"),
}
CV_MODELS = ["MoViT-SAM", "Hybrid+Adam", "Hybrid+H-SAM", "EfficientNetV2"]
REFERENCE  = "MoViT-SAM"

def n_params(m):
    return (int(sum(np.prod(w.shape) for w in m.trainable_weights)),
            int(sum(np.prod(w.shape) for w in m.weights)))

def inner_split(idx, seed=SEED):
    """Carve a patient-disjoint early-stopping set out of the TRAINING indices only."""
    gss = GroupShuffleSplit(n_splits=1, test_size=INNER_VAL_FRAC, random_state=seed)
    a, b = next(gss.split(idx, labels[idx], groups[idx]))
    return idx[a], idx[b]

def fit(name, tr_idx, es_idx, seed=SEED, path_arr=None, label_arr=None, cw_idx=None):
    keras.backend.clear_session(); tf.random.set_seed(seed); np.random.seed(seed)
    model = MODELS[name]()
    cw = class_weights_for(cw_idx if cw_idx is not None else tr_idx) if path_arr is None else \
         {k: float(v) for k, v in zip(range(NUM_CLASSES),
          compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=label_arr[tr_idx]))}
    tr = make_ds(tr_idx, training=True, cw=cw, path_arr=path_arr, label_arr=label_arr)
    es = make_ds(es_idx, path_arr=path_arr, label_arr=label_arr)
    model.fit(tr, validation_data=es, epochs=EPOCHS, verbose=0, callbacks=[
        callbacks.EarlyStopping("val_accuracy", patience=PATIENCE, restore_best_weights=True),
        callbacks.ReduceLROnPlateau("val_loss", factor=0.5, patience=3, min_lr=1e-6)])
    return model

def predict(model, idx, path_arr=None, label_arr=None):
    out = [model(x, training=False).numpy() for x, _, _ in make_ds(idx, path_arr=path_arr, label_arr=label_arr)]
    return np.vstack(out)

def macro_auc(y, p):
    try: return float(roc_auc_score(np.eye(NUM_CLASSES)[y], p, average="macro", multi_class="ovr"))
    except ValueError: return float("nan")

def metrics(y, p):
    pred = p.argmax(1)
    return {"accuracy": float((pred == y).mean()), "macro_auc": macro_auc(y, p),
            "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0))}

In [ ]:
# ============================================================
# 10b - Three more frozen backbones; all nine models enter the cross-validation
# ============================================================
# Same protocol as the six models of Section 10: pretrained backbone frozen, raw [0,255] input,
# backbone-specific preprocessing inside the model, identical MLP head, Adam, focal loss.
def _frozen_app_features(app, preprocess, dim, name):
    if TINY_TEST: return _tiny_features(dim, name)
    inp = keras.Input((IMG_SIZE, IMG_SIZE, CHANNELS))
    bb = app(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS))
    bb.trainable = False
    x = layers.GlobalAveragePooling2D()(bb(preprocess(inp), training=False))
    return keras.Model(inp, x, name=name)

def resnet_features(name="res"):
    return _frozen_app_features(keras.applications.ResNet50,
                                keras.applications.resnet50.preprocess_input, 2048, name)

def densenet_features(name="dense"):
    return _frozen_app_features(keras.applications.DenseNet121,
                                keras.applications.densenet.preprocess_input, 1024, name)

def dino_features(name="dino"):
    """Frozen DINO ViT-B/16 (self-supervised, ImageNet-1k) -> 768-d CLS token. It uses the ViT
    architecture, so it loads through TFViTModel exactly like the ViT-B/16 stream. Its
    preprocessor_config.json specifies the ImageNet mean and standard deviation."""
    if TINY_TEST: return _tiny_features(768, name)
    from transformers import TFViTModel
    dino = TFViTModel.from_pretrained("facebook/dino-vitb16")
    dino.trainable = False
    inp = keras.Input((IMG_SIZE, IMG_SIZE, CHANNELS))
    x = (layers.Rescaling(1.0 / 255)(inp) - IMAGENET_MEAN) / IMAGENET_STD
    x = layers.Permute((3, 1, 2))(x)
    return keras.Model(inp, dino(x).last_hidden_state[:, 0, :], name=name)

def build_frozen(feat_fn, lr, tag):
    """Same head and wrapper as build_single()."""
    f = feat_fn()
    x = layers.Dropout(0.3)(layers.Dense(256, activation="relu")(f.output))
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    m = SharpnessAwareModel(keras.Model(f.input, out, name=f"{tag}_single"), mode="none",
                            name=f"{tag}_adam")
    _ = m(tf.zeros((1, IMG_SIZE, IMG_SIZE, CHANNELS)))
    return _compile(m, lr)

MODELS.update({
    "ResNet50":    lambda: build_frozen(resnet_features, LR_MOBILENET, "resnet50"),
    "DenseNet121": lambda: build_frozen(densenet_features, LR_MOBILENET, "densenet121"),
    "DINO ViT-B/16": lambda: build_frozen(dino_features, LR_VIT, "dino_vitb16"),
})
MODEL_FAMILY = {"MobileNetV2": "CNN", "EfficientNetV2": "CNN", "ResNet50": "CNN", "DenseNet121": "CNN",
                "ViT-B/16": "Transformer", "DINO ViT-B/16": "Transformer",
                "Hybrid+Adam": "Hybrid", "MoViT-SAM": "Hybrid", "Hybrid+H-SAM": "Hybrid"}
# Order in which the models are trained.
CMP_MODELS = ["MoViT-SAM", "Hybrid+Adam", "Hybrid+H-SAM", "EfficientNetV2",
              "MobileNetV2", "ResNet50", "DenseNet121", "ViT-B/16", "DINO ViT-B/16"]

# Build each added model now, train it for ONE batch and predict once. A model that cannot be
# built or trained is left out here instead of stopping the long cross-validation half-way.
CMP_DROPPED = {}
_chk = np.arange(min(8, len(paths)))
for _n in ["ResNet50", "DenseNet121", "DINO ViT-B/16"]:
    try:
        keras.backend.clear_session()
        _m = MODELS[_n]()
        _m.fit(make_ds(_chk, training=True, cw=class_weights_for(_chk), batch=4),
               epochs=1, steps_per_epoch=1, verbose=0)
        assert predict(_m, _chk[:4]).shape == (len(_chk[:4]), NUM_CLASSES)
        print(f"{_n}: builds OK and trains, (trainable, total) params = {n_params(_m)}"); del _m
    except Exception as ex:
        CMP_DROPPED[_n] = f"{type(ex).__name__}: {str(ex)[:600]}"
        print(f"!!! {_n} could not be built or trained and is left out: {CMP_DROPPED[_n]}")
CV_MODELS = [m for m in CMP_MODELS if m not in CMP_DROPPED]
print("Models in the cross-validation:", CV_MODELS)

CMP_DIR = RESULTS_DIR / "comparative"; CMP_DIR.mkdir(exist_ok=True)
CMP_T0 = time.time()        # clock for the time guard in Section 16b
CMP_LEAK_DEADLINE_H = 8.5   # Kaggle stops a run at 12 h and then keeps nothing

def cmp_package():
    """Zip the result files. Called after each analysis step, so a partial run still
    leaves a usable file."""
    import zipfile
    out = Path("comparative_results.zip")
    keep = [p for p in RESULTS_DIR.iterdir() if p.is_file() and p.suffix in {".csv", ".json", ".npy"}]
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for p in keep: z.write(p, f"results/{p.name}")
        for p in CMP_DIR.iterdir():
            if p.is_file(): z.write(p, f"comparative/{p.name}")
    print(f"Packaged {len(keep)} result files + comparative tables -> {out.resolve()}")

## Cross-validation on identical patient-level folds

In [ ]:
# ============================================================
# 11 - Five-fold stratified-group CV on all originals (identical folds for every model)
# ============================================================
sgkf = StratifiedGroupKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(sgkf.split(paths, labels, groups))
for k, (tr, va) in enumerate(FOLDS):
    assert set(groups[tr]).isdisjoint(groups[va])

fold_rows, OOF = [], {}
for name in CV_MODELS:
    oof = np.zeros((len(df), NUM_CLASSES), np.float32)
    for k, (tr, va) in enumerate(FOLDS, 1):
        ck = CKPT_DIR / f"{re.sub(r'[^A-Za-z0-9]+', '_', name)}_fold{k}.npz"
        if ck.exists():
            p = np.load(ck)["prob"]
        else:
            t0 = time.time()
            tr_in, es_in = inner_split(tr, seed=SEED + k)
            model = fit(name, tr_in, es_in, seed=SEED + k, cw_idx=tr_in)
            p = predict(model, va)
            np.savez(ck, prob=p, idx=va)
            print(f"[{name}] fold {k}: {(time.time() - t0) / 60:.1f} min")
        oof[va] = p
        fold_rows.append({"model": name, "fold": k, **metrics(labels[va], p)})
    OOF[name] = oof
    np.save(RESULTS_DIR / f"oof_{re.sub(r'[^A-Za-z0-9]+', '_', name)}.npy", oof)

folds_df = pd.DataFrame(fold_rows)
folds_df.to_csv(RESULTS_DIR / "cv_per_fold_all_models.csv", index=False)
print(folds_df.to_string(index=False))

## Analysis of the out-of-fold predictions

In [ ]:
# ============================================================
# 12 - Reference model: per-fold summary, fold-level paired t-tests, pooled out-of-fold metrics
# ============================================================
def mean_sd_ci(x):
    x = np.asarray(x, float); m, sd = x.mean(), x.std(ddof=1)
    lo, hi = sp_stats.t.interval(0.95, len(x) - 1, loc=m, scale=sd / np.sqrt(len(x)))
    return m, sd, lo, hi

t3 = folds_df[folds_df.model == REFERENCE][["fold", "accuracy", "macro_auc", "macro_f1"]].copy()
summ = {c: mean_sd_ci(t3[c]) for c in ["accuracy", "macro_auc", "macro_f1"]}
t3 = t3.round(4).astype(object)
t3.loc[len(t3)] = ["Mean ± SD", *[f"{summ[c][0]:.4f} ± {summ[c][1]:.4f}" for c in summ]]
t3.loc[len(t3)] = ["95% CI (t)", *[f"[{summ[c][2]:.3f}, {summ[c][3]:.3f}]" for c in summ]]
print("Reference model, five-fold patient-level CV\n", t3.to_string(index=False))
t3.to_csv(RESULTS_DIR / "table3_movit_sam_folds.csv", index=False)

base = folds_df[folds_df.model == REFERENCE].sort_values("fold")
others = [m for m in CV_MODELS if m != REFERENCE]
t4 = []
for name in CV_MODELS:
    d = folds_df[folds_df.model == name].sort_values("fold")
    row = {"Model": name, "Acc mean": d.accuracy.mean(), "Acc SD": d.accuracy.std(ddof=1),
           "Macro-AUC": d.macro_auc.mean(), "Macro-F1": d.macro_f1.mean()}
    if name != REFERENCE:
        p = sp_stats.ttest_rel(base.accuracy.values, d.accuracy.values).pvalue
        row["p (paired t)"] = p
        row["p (Bonferroni)"] = min(1.0, p * len(others))
    t4.append(row)
t4 = pd.DataFrame(t4).round(4)
print("Fold-level comparison with the reference model (paired t-test over folds; Bonferroni for",
      len(others), "comparisons)\n", t4.to_string(index=False))
t4.to_csv(RESULTS_DIR / "table4_cv_models.csv", index=False)

def bootstrap_ci(y, p, fn, n=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        v = fn(y[i], p[i])
        if not np.isnan(v): vals.append(v)
    return np.percentile(vals, [2.5, 97.5])

P_oof = OOF[REFERENCE]
pooled = metrics(labels, P_oof)
acc_ci = bootstrap_ci(labels, P_oof, lambda y, p: float((p.argmax(1) == y).mean()))
auc_ci = bootstrap_ci(labels, P_oof, macro_auc)

pt = pd.DataFrame(P_oof).groupby(groups).mean()
pt_y = pd.Series(labels).groupby(groups).first().loc[pt.index].values
patient_level = metrics(pt_y, pt.values)

POOLED = {"pooled_oof": pooled, "acc_ci": acc_ci.tolist(), "macro_auc_ci": auc_ci.tolist(),
          "patient_level": patient_level, "n_images": len(df), "n_patients": int(len(pt))}
print(json.dumps(POOLED, indent=2))
(RESULTS_DIR / "pooled_oof_summary.json").write_text(json.dumps(POOLED, indent=2))

In [ ]:
# ============================================================
# 13 - Reference model: per-class performance, confusion matrix, ROC and precision-recall curves
# ============================================================
pred_oof = P_oof.argmax(1)
rep = classification_report(labels, pred_oof, labels=range(NUM_CLASSES), target_names=CLASS_NAMES,
                            output_dict=True, zero_division=0)
Y1 = np.eye(NUM_CLASSES)[labels]
roc_c, pr_c = {}, {}
for i, c in enumerate(CLASS_NAMES):
    roc_c[c] = roc_auc_score(Y1[:, i], P_oof[:, i]) if Y1[:, i].any() else np.nan
    pr_c[c]  = average_precision_score(Y1[:, i], P_oof[:, i]) if Y1[:, i].any() else np.nan
t5 = pd.DataFrame([{"Class": c, "Precision": rep[c]["precision"], "Recall": rep[c]["recall"],
                    "F1": rep[c]["f1-score"], "Support": int(rep[c]["support"]),
                    "ROC-AUC": roc_c[c], "PR-AUC": pr_c[c]} for c in CLASS_NAMES])
t5.loc[len(t5)] = ["Macro avg", rep["macro avg"]["precision"], rep["macro avg"]["recall"],
                   rep["macro avg"]["f1-score"], len(df), np.nanmean(list(roc_c.values())),
                   np.nanmean(list(pr_c.values()))]
print("Per-class performance of the reference model\n", t5.round(3).to_string(index=False))
t5.round(4).to_csv(RESULTS_DIR / "table5_per_class.csv", index=False)

cm = confusion_matrix(labels, pred_oof, labels=range(NUM_CLASSES))
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax[0], xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
sns.heatmap(cm / cm.sum(1, keepdims=True).clip(min=1) * 100, annot=True, fmt=".1f", cmap="Blues",
            ax=ax[1], xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
for a, t in zip(ax, ["Counts", "Row %"]): a.set(title=t, xlabel="Predicted", ylabel="True")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "fig3_confusion.png", dpi=300); plt.close()

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for i, c in enumerate(CLASS_NAMES):
    if not Y1[:, i].any(): continue
    f, t, _ = roc_curve(Y1[:, i], P_oof[:, i]); ax[0].plot(f, t, label=f"{c} ({roc_c[c]:.3f})")
    pr, rc, _ = precision_recall_curve(Y1[:, i], P_oof[:, i]); ax[1].plot(rc, pr, label=f"{c} ({pr_c[c]:.3f})")
ax[0].plot([0, 1], [0, 1], "k--"); ax[0].set(xlabel="1 - Specificity", ylabel="Sensitivity", title="ROC (pooled OOF)")
ax[1].set(xlabel="Recall", ylabel="Precision", title="Precision-recall (pooled OOF)")
for a in ax: a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "fig5_roc_pr.png", dpi=300); plt.close()

In [ ]:
# ============================================================
# 14 - Reference model: cross-fitted temperature scaling, ECE, reliability diagram
# ============================================================
# For each fold k, T is fitted by minimising NLL on the OOF predictions of the OTHER folds and
# applied to fold k, so no prediction is calibrated with a temperature fitted on itself.
def apply_T(p, T):
    z = np.log(np.clip(p, 1e-12, 1.0)) / T
    z -= z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def fit_T(p, y):
    nll = lambda T: -np.mean(np.log(np.clip(apply_T(p, T)[np.arange(len(y)), y], 1e-12, 1)))
    return float(minimize_scalar(nll, bounds=(0.05, 10.0), method="bounded").x)

def ece(p, y, n_bins=10):
    conf, pred = p.max(1), p.argmax(1)
    edges = np.linspace(0, 1, n_bins + 1); e = 0.0; pts = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            acc, cf = (pred[m] == y[m]).mean(), conf[m].mean()
            e += m.mean() * abs(acc - cf); pts.append((cf, acc))
    return float(e), np.array(pts)

P_cal = np.zeros_like(P_oof); Ts = []
for tr, va in FOLDS:
    T = fit_T(P_oof[tr], labels[tr]); Ts.append(T)
    P_cal[va] = apply_T(P_oof[va], T)
ece_before, pts_b = ece(P_oof, labels)
ece_after,  pts_a = ece(P_cal, labels)
CAL = {"T_per_fold": Ts, "T_mean": float(np.mean(Ts)), "ece_before": ece_before, "ece_after": ece_after,
       "argmax_unchanged": bool((P_cal.argmax(1) == P_oof.argmax(1)).all())}
print(json.dumps(CAL, indent=2))
(RESULTS_DIR / "calibration.json").write_text(json.dumps(CAL, indent=2))

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.plot(pts_b[:, 0], pts_b[:, 1], "o-", color="tab:orange", label=f"Before (ECE={ece_before:.3f})")
ax.plot(pts_a[:, 0], pts_a[:, 1], "s-", color="tab:green", label=f"After, cross-fitted T (ECE={ece_after:.3f})")
ax.set(xlabel="Mean predicted confidence", ylabel="Empirical accuracy", xlim=(0, 1), ylim=(0, 1))
ax.legend(); plt.tight_layout(); plt.savefig(RESULTS_DIR / "fig4_reliability.png", dpi=300); plt.close()

In [ ]:
# ============================================================
# 14b - Comparison of all nine models on the same folds
# ============================================================
# Uses only the stored out-of-fold predictions, so it can be re-run without training.
CMP_DONE = [m for m in CV_MODELS if m in OOF]

def cmp_patient_level(P):
    """One prediction per patient: mean softmax over the patient's images."""
    pt = pd.DataFrame(P).groupby(groups).mean()
    y = pd.Series(labels).groupby(groups).first().loc[pt.index].values
    return y, pt.values

def cmp_cross_fitted_T(P):
    """Cross-fitted temperature scaling, exactly as in Section 14."""
    Pc = np.zeros_like(P); Ts = []
    for tr, va in FOLDS:
        T = fit_T(P[tr], labels[tr]); Ts.append(T); Pc[va] = apply_T(P[va], T)
    return Pc, float(np.mean(Ts))

def cmp_brier(p, y):
    return float(np.mean(np.sum((p - np.eye(NUM_CLASSES)[y]) ** 2, axis=1)))

def cmp_holm(pvals):
    """Holm step-down adjusted p-values."""
    p = np.asarray(pvals, float); m = len(p); adj = np.empty(m); running = 0.0
    for rank, j in enumerate(np.argsort(p)):
        running = max(running, (m - rank) * p[j]); adj[j] = min(1.0, running)
    return adj

def cmp_mcnemar(y, pa, pb):
    """Exact McNemar test on paired predictions (two-sided binomial on discordant pairs)."""
    a_ok, b_ok = pa == y, pb == y
    n10, n01 = int(np.sum(a_ok & ~b_ok)), int(np.sum(~a_ok & b_ok))
    p = 1.0 if n10 + n01 == 0 else float(sp_stats.binomtest(min(n10, n01), n10 + n01, 0.5).pvalue)
    return n10, n01, p

# ---------------- Main comparison and per-class results ----------------
acc_fn = lambda y, p: float((p.argmax(1) == y).mean())
c1, c2, PT = [], [], {}
for name in CMP_DONE:
    P = OOF[name]; pred = P.argmax(1); d = folds_df[folds_df.model == name]
    acc_ci, auc_ci = bootstrap_ci(labels, P, acc_fn), bootstrap_ci(labels, P, macro_auc)
    rep = classification_report(labels, pred, labels=list(range(NUM_CLASSES)),
                                target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    py, pp = cmp_patient_level(P); PT[name] = (py, pp)
    Pc, T_mean = cmp_cross_fitted_T(P)
    is_mpox, said_mpox = labels == MPOX_IDX, pred == MPOX_IDX
    c1.append({
        "Model": name, "Family": MODEL_FAMILY.get(name, ""),
        "Fold acc mean": d.accuracy.mean(), "Fold acc SD": d.accuracy.std(ddof=1),
        "OOF acc": acc_fn(labels, P), "OOF acc CI low": acc_ci[0], "OOF acc CI high": acc_ci[1],
        "Macro-AUC": macro_auc(labels, P), "AUC CI low": auc_ci[0], "AUC CI high": auc_ci[1],
        "Macro-F1": rep["macro avg"]["f1-score"], "Balanced acc": rep["macro avg"]["recall"],
        "Mpox sensitivity": float(said_mpox[is_mpox].mean()),
        "Mpox specificity": float((~said_mpox[~is_mpox]).mean()),
        "Patient acc": acc_fn(py, pp), "Patient macro-AUC": macro_auc(py, pp),
        "ECE": ece(P, labels)[0], "ECE after T": ece(Pc, labels)[0], "T mean": T_mean,
        "Brier": cmp_brier(P, labels), "N images": len(labels), "N patients": len(py)})
    for c in CLASS_NAMES:
        c2.append({"Model": name, "Class": c, "Precision": rep[c]["precision"],
                   "Recall": rep[c]["recall"], "F1": rep[c]["f1-score"], "Support": int(rep[c]["support"])})
cmp_main = pd.DataFrame(c1).sort_values("OOF acc", ascending=False).reset_index(drop=True)
cmp_main.round(4).to_csv(CMP_DIR / "cmp_table_main.csv", index=False)
pd.DataFrame(c2).round(4).to_csv(CMP_DIR / "cmp_table_perclass.csv", index=False)
print("Main comparison - all models, identical patient-level folds, pooled out-of-fold predictions")
print(cmp_main[["Model", "Family", "Fold acc mean", "Fold acc SD", "OOF acc", "Macro-AUC", "Macro-F1",
                "Patient acc", "ECE", "ECE after T"]].round(4).to_string(index=False))
fam = cmp_main.groupby("Family")[["OOF acc", "Macro-AUC", "Macro-F1"]].agg(["mean", "min", "max"]).round(4)
fam.to_csv(CMP_DIR / "cmp_table_family.csv")
print("\nBy family:\n", fam.to_string())

# ---------------- Friedman test over folds ----------------
piv = folds_df[folds_df.model.isin(CMP_DONE)].pivot(index="fold", columns="model", values="accuracy")[CMP_DONE]
piv.round(4).to_csv(CMP_DIR / "cmp_fold_accuracy.csv")
mean_rank = piv.rank(axis=1, ascending=False).mean()
fr = pd.DataFrame({"Model": mean_rank.index, "Mean rank (1 = best)": mean_rank.values.round(3)})
try:
    chi2, p_fr = sp_stats.friedmanchisquare(*[piv[m].values for m in CMP_DONE])
    fr["Friedman chi2"], fr["Friedman p"], fr["k models"], fr["n folds"] = chi2, p_fr, len(CMP_DONE), len(piv)
    print(f"\nFriedman test over {len(piv)} folds, {len(CMP_DONE)} models: chi2 = {chi2:.3f}, p = {p_fr:.4f}")
except Exception as ex:
    print("\nFriedman test not computed:", repr(ex)[:120])
fr.sort_values("Mean rank (1 = best)").to_csv(CMP_DIR / "cmp_friedman.csv", index=False)
print(fr.sort_values("Mean rank (1 = best)")[["Model", "Mean rank (1 = best)"]].to_string(index=False))

# ---------------- Pairwise McNemar tests, Holm-adjusted ----------------
# Reported at image level (n images) and at patient level (one prediction per patient,
# so the paired observations are independent).
pairs = []
for a, b in itertools.combinations(CMP_DONE, 2):
    i10, i01, ip = cmp_mcnemar(labels, OOF[a].argmax(1), OOF[b].argmax(1))
    q10, q01, qp = cmp_mcnemar(PT[a][0], PT[a][1].argmax(1), PT[b][1].argmax(1))
    pairs.append({"Model A": a, "Model B": b,
                  "Image: A right, B wrong": i10, "Image: A wrong, B right": i01,
                  "Image: acc A - acc B": (i10 - i01) / len(labels), "Image: p": ip,
                  "Patient: A right, B wrong": q10, "Patient: A wrong, B right": q01,
                  "Patient: acc A - acc B": (q10 - q01) / len(PT[a][0]), "Patient: p": qp})
cmp_pairs = pd.DataFrame(pairs)
if len(cmp_pairs):
    cmp_pairs["Image: p (Holm)"] = cmp_holm(cmp_pairs["Image: p"].values)
    cmp_pairs["Patient: p (Holm)"] = cmp_holm(cmp_pairs["Patient: p"].values)
    cmp_pairs.to_csv(CMP_DIR / "cmp_pairwise_mcnemar.csv", index=False)
    sig = cmp_pairs[cmp_pairs["Patient: p (Holm)"] < 0.05]
    print(f"\nPairwise McNemar tests - {len(cmp_pairs)} model pairs; significant at patient level after Holm: {len(sig)}")
    if len(sig):
        print(sig[["Model A", "Model B", "Patient: acc A - acc B", "Patient: p (Holm)"]].round(4).to_string(index=False))

# arrays needed to redraw any figure later without re-training
np.save(CMP_DIR / "labels.npy", labels); np.save(CMP_DIR / "groups.npy", groups.astype(str))
fold_id = np.zeros(len(labels), int)
for k, (_, va) in enumerate(FOLDS, 1): fold_id[va] = k
np.save(CMP_DIR / "fold_id.npy", fold_id)
(CMP_DIR / "cmp_run_info.json").write_text(json.dumps(
    {"models": CMP_DONE, "dropped": CMP_DROPPED, "k_folds": K_FOLDS, "epochs": EPOCHS, "patience": PATIENCE,
     "inner_val_frac": INNER_VAL_FRAC, "n_boot": N_BOOT, "seed": SEED, "tiny_test": TINY_TEST,
     "smoke_test": SMOKE_TEST, "hardware": HARDWARE}, indent=2, default=str))
cmp_package()

In [ ]:
# ============================================================
# 15 - Reference model: binary Mpox screening, PPV/NPV, decision curve analysis, error taxonomy
# ============================================================
yb = (labels == MPOX_IDX).astype(int)
sb = P_cal[:, MPOX_IDX]
b_auc = roc_auc_score(yb, sb)
b_auc_ci = bootstrap_ci(yb, sb, lambda y, s: roc_auc_score(y, s) if 0 < y.sum() < len(y) else np.nan)
fpr, tpr, thr = roc_curve(yb, sb)

arg_pos = (P_cal.argmax(1) == MPOX_IDX)
sens_arg = (arg_pos & (yb == 1)).sum() / yb.sum()
spec_arg = (~arg_pos & (yb == 0)).sum() / (yb == 0).sum()
ops = [{"operating point": "argmax", "sensitivity": sens_arg, "specificity": spec_arg}]
for s in SENS_TARGETS:
    i = int(np.argmax(tpr >= s))
    ops.append({"operating point": f"sens>={s:.2f}", "threshold": thr[i],
                "sensitivity": tpr[i], "specificity": 1 - fpr[i]})
ops = pd.DataFrame(ops); print(ops.round(3).to_string(index=False))
ops.to_csv(RESULTS_DIR / "screening_operating_points.csv", index=False)

def ppv_npv(se, sp, prev):
    ppv = se * prev / (se * prev + (1 - sp) * (1 - prev))
    npv = sp * (1 - prev) / (sp * (1 - prev) + (1 - se) * prev)
    return ppv, npv

prev_list = PREVALENCES + [float(yb.mean())]
t7 = pd.DataFrame([{"Prevalence": f"{p*100:.1f}%", **dict(zip(["PPV", "NPV"], ppv_npv(sens_arg, spec_arg, p)))}
                   for p in prev_list]).round(3)
print("PPV and NPV at the predicted-class operating point\n", t7.to_string(index=False))
t7.to_csv(RESULTS_DIR / "table7_ppv_npv.csv", index=False)

def net_benefit(y, s, pt):
    pos = s >= pt; n = len(y)
    tp, fp = (pos & (y == 1)).sum(), (pos & (y == 0)).sum()
    return tp / n - fp / n * pt / (1 - pt)
prev = yb.mean()
dca = pd.DataFrame({"threshold": DCA_THRESHOLDS,
                    "model": [net_benefit(yb, sb, t) for t in DCA_THRESHOLDS],
                    "treat_all": [prev - (1 - prev) * t / (1 - t) for t in DCA_THRESHOLDS],
                    "treat_none": 0.0})
dca.to_csv(RESULTS_DIR / "dca.csv", index=False)
better = dca[(dca.model > dca.treat_all) & (dca.model > 0)].threshold
DCA_RANGE = (float(better.min()), float(better.max())) if len(better) else None
print("Net benefit above treat-all and treat-none for thresholds:", DCA_RANGE,
      "(check the range is contiguous in dca.csv)")
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(dca.threshold, dca.model, "b-", lw=2, label="MoViT-SAM")
ax.plot(dca.threshold, dca.treat_all, "k--", label="Screen all")
ax.axhline(0, color="k", lw=1, label="Screen none")
ax.set(xlabel="Threshold probability", ylabel="Net benefit", ylim=(-0.05, max(0.05, prev * 1.1)))
ax.legend(); plt.tight_layout(); plt.savefig(RESULTS_DIR / "fig6_dca.png", dpi=300); plt.close()

err = pred_oof != labels
dangerous = int(((labels == MPOX_IDX) & (pred_oof == CLASS_NAMES.index("Healthy"))).sum())
moderate  = int((err & ((labels == MPOX_IDX) | (pred_oof == MPOX_IDX))).sum()) - dangerous
benign    = int((err & (labels != MPOX_IDX) & (pred_oof != MPOX_IDX)).sum())
CLIN = {"binary_auc": b_auc, "binary_auc_ci": b_auc_ci.tolist(), "sens_argmax": sens_arg,
        "spec_argmax": spec_arg, "dca_positive_range": DCA_RANGE,
        "errors": {"dangerous_mpox_to_healthy": dangerous, "moderate": moderate, "benign": benign,
                   "total": int(err.sum())}}
print(json.dumps(CLIN, indent=2, default=float))
(RESULTS_DIR / "clinical.json").write_text(json.dumps(CLIN, indent=2, default=float))

## Leakage experiments

In [ ]:
# ============================================================
# 16 - Leakage quantification for the reference model
# ============================================================
# Protocol A: image-level stratified 85/15 split that ignores patient identity.
# Protocol B: 6 augmented copies per original are materialised first (755 x 7 = 5,285 images),
#             then split 85/15 at image level: the leakage pattern of prior MSLD v2.0 studies.
# Both use the MoViT-SAM architecture and training recipe; early stopping uses an inner split
# drawn in the same (leaky) way from the training portion, as a naive pipeline would.
leak_rows = [{"Protocol": "Patient-level 5-fold CV (leakage-free)", "Contamination": "0 patients",
              "Accuracy": pooled["accuracy"], "Macro-AUC": pooled["macro_auc"]}]

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
a_tr, a_te = next(sss.split(paths, labels))
a_tr_in, a_es = next(StratifiedShuffleSplit(n_splits=1, test_size=INNER_VAL_FRAC,
                                            random_state=SEED).split(paths[a_tr], labels[a_tr]))
mA = fit(REFERENCE, a_tr[a_tr_in], a_tr[a_es], cw_idx=a_tr[a_tr_in])
pA = predict(mA, a_te)
shared_pat = len(set(groups[a_tr]) & set(groups[a_te]))
mA_ = metrics(labels[a_te], pA)
leak_rows.append({"Protocol": "Naive image-level split (patient overlap)",
                  "Contamination": f"{shared_pat} patients",
                  "Accuracy": mA_["accuracy"], "Macro-AUC": mA_["macro_auc"]})

AUG_DIR = Path(os.environ.get("MOVIT_AUG_DIR", "./protocolB_images")); AUG_DIR.mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)
b_paths, b_labels, b_src = [], [], []
for i, p in enumerate(paths):
    img = np.array(Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
    b_paths.append(p); b_labels.append(labels[i]); b_src.append(i)
    for c in range(AUG_COPIES_B):
        ang, sc = rng.uniform(-15, 15), rng.uniform(0.9, 1.1)
        M = cv2.getRotationMatrix2D((IMG_SIZE / 2, IMG_SIZE / 2), ang, sc)
        out = cv2.warpAffine(img, M, (IMG_SIZE, IMG_SIZE), borderMode=cv2.BORDER_REFLECT)
        if rng.random() < 0.5: out = out[:, ::-1]
        q = AUG_DIR / f"{i:05d}_{c}.jpg"
        if not q.exists(): Image.fromarray(np.ascontiguousarray(out)).save(q, quality=95)
        b_paths.append(str(q)); b_labels.append(labels[i]); b_src.append(i)
b_paths, b_labels, b_src = np.array(b_paths), np.array(b_labels), np.array(b_src)
print("Protocol B pool:", len(b_paths), "images")

rs = np.random.default_rng(SEED).permutation(len(b_paths))
n_te = int(round(0.15 * len(b_paths)))
b_te, b_tr = rs[:n_te], rs[n_te:]
b_tr_in, b_es = b_tr[:int(len(b_tr) * (1 - INNER_VAL_FRAC))], b_tr[int(len(b_tr) * (1 - INNER_VAL_FRAC)):]
leaked = np.isin(b_src[b_te], b_src[b_tr])
mB = fit(REFERENCE, b_tr_in, b_es, path_arr=b_paths, label_arr=b_labels)
pB = predict(mB, b_te, path_arr=b_paths, label_arr=b_labels)
mB_ = metrics(b_labels[b_te], pB)
leak_rows.append({"Protocol": "Augment-before-split (prior-study style)",
                  "Contamination": f"{int(leaked.sum())}/{len(b_te)} test images ({leaked.mean()*100:.1f}%)",
                  "Accuracy": mB_["accuracy"], "Macro-AUC": mB_["macro_auc"]})

t2 = pd.DataFrame(leak_rows)
t2["Inflation vs clean (pp)"] = (t2.Accuracy - t2.Accuracy.iloc[0]) * 100
print("Leakage protocols, reference model\n", t2.round(4).to_string(index=False))
t2.round(4).to_csv(RESULTS_DIR / "table2_leakage.csv", index=False)

In [ ]:
# ============================================================
# 16b - Leakage inflation by architecture
# ============================================================
# Repeats Protocols A and B of Section 16, with exactly the same splits, for one CNN and one
# transformer. MoViT-SAM (hybrid) is taken from Section 16. The clean reference for each model
# is its own pooled out-of-fold result.
CMP_LEAK_MODELS = [m for m in ["MobileNetV2", "ViT-B/16"] if m in OOF]
_leak_file = CMP_DIR / "cmp_leakage_runs.csv"
_runs = pd.read_csv(_leak_file).to_dict("records") if _leak_file.exists() else []

def _leak_done(model, proto):
    return any(r["model"] == model and r["protocol"] == proto for r in _runs)

def _time_left():
    """False once the run is too close to Kaggle's 12-hour limit to start another leakage run."""
    h = (time.time() - CMP_T0) / 3600
    if h > CMP_LEAK_DEADLINE_H:
        print(f"Time guard: {h:.1f} h elapsed, remaining leakage runs skipped to protect the run.")
        return False
    return True

for name in CMP_LEAK_MODELS:
    if not _leak_done(name, "A") and _time_left():
        m = fit(name, a_tr[a_tr_in], a_tr[a_es], cw_idx=a_tr[a_tr_in])
        r = metrics(labels[a_te], predict(m, a_te)); del m
        _runs.append({"model": name, "protocol": "A", "accuracy": r["accuracy"], "macro_auc": r["macro_auc"]})
        pd.DataFrame(_runs).to_csv(_leak_file, index=False)
        print(f"[{name}] Protocol A: acc = {r['accuracy']:.4f}, AUC = {r['macro_auc']:.4f}")
    if not _leak_done(name, "B") and _time_left():
        m = fit(name, b_tr_in, b_es, path_arr=b_paths, label_arr=b_labels)
        r = metrics(b_labels[b_te], predict(m, b_te, path_arr=b_paths, label_arr=b_labels)); del m
        _runs.append({"model": name, "protocol": "B", "accuracy": r["accuracy"], "macro_auc": r["macro_auc"]})
        pd.DataFrame(_runs).to_csv(_leak_file, index=False)
        print(f"[{name}] Protocol B: acc = {r['accuracy']:.4f}, AUC = {r['macro_auc']:.4f}")

_labelA = f"{shared_pat} patients"
_labelB = f"{int(leaked.sum())}/{len(b_te)} test images ({leaked.mean() * 100:.1f}%)"
c5 = []
def _add(model, proto, contam, acc, auc_v, clean_acc):
    c5.append({"Model": model, "Family": MODEL_FAMILY.get(model, ""), "Protocol": proto,
               "Contamination": contam, "Accuracy": acc, "Macro-AUC": auc_v,
               "Inflation vs clean (pp)": (acc - clean_acc) * 100})
_add(REFERENCE, "Patient-level 5-fold CV (leakage-free)", "0 patients", pooled["accuracy"],
     pooled["macro_auc"], pooled["accuracy"])
_add(REFERENCE, "A: naive image-level split", _labelA, mA_["accuracy"], mA_["macro_auc"], pooled["accuracy"])
_add(REFERENCE, "B: augment-before-split", _labelB, mB_["accuracy"], mB_["macro_auc"], pooled["accuracy"])
for name in CMP_LEAK_MODELS:
    clean = metrics(labels, OOF[name])
    _add(name, "Patient-level 5-fold CV (leakage-free)", "0 patients", clean["accuracy"],
         clean["macro_auc"], clean["accuracy"])
    for proto, lab, contam in (("A", "A: naive image-level split", _labelA),
                               ("B", "B: augment-before-split", _labelB)):
        r = next((x for x in _runs if x["model"] == name and x["protocol"] == proto), None)
        if r: _add(name, lab, contam, r["accuracy"], r["macro_auc"], clean["accuracy"])
cmp_leak = pd.DataFrame(c5)
cmp_leak.round(4).to_csv(CMP_DIR / "cmp_table_leakage.csv", index=False)
print("Leakage inflation by architecture\n", cmp_leak.round(4).to_string(index=False))
cmp_package()

## Results

In [ ]:
# ============================================================
# 17 - Package the results
# ============================================================
cmp_package()
for f in sorted(CMP_DIR.iterdir()): print("  ", f.name)